# Beginner: Python Fundamentals You'll Use Constantly in AI Work

**Sub-topic 1.3 — Python Refresher for AI**

Covers list/dict comprehensions, unpacking, `*args`/`**kwargs`, and a simple class — all demonstrated on small text-processing tasks, because "process a batch of text" is the single most common operation in GenAI code.

In [1]:
DOCUMENTS = [
    "the quick brown fox jumps over the lazy dog",
    "machine learning models learn patterns from data",
    "generative ai models generate new content from patterns",
    "the lazy dog sleeps while the fox jumps",
]

## List comprehensions

Build a new list from an existing iterable in one line. You'll use this constantly to transform API results or dataset rows.

In [2]:
word_counts = [len(doc.split()) for doc in DOCUMENTS]
print(f"Word count per document: {word_counts}")

long_docs = [doc for doc in DOCUMENTS if len(doc.split()) > 7]
print(f"Documents with >7 words: {long_docs}")

upper_first_word = [doc.split()[0].upper() for doc in DOCUMENTS]
print(f"First word of each doc, uppercased: {upper_first_word}")

Word count per document: [9, 7, 8, 8]
Documents with >7 words: ['the quick brown fox jumps over the lazy dog', 'generative ai models generate new content from patterns', 'the lazy dog sleeps while the fox jumps']
First word of each doc, uppercased: ['THE', 'MACHINE', 'GENERATIVE', 'THE']


## Dict comprehensions

Build a lookup table in one line. This is exactly the pattern you'll use to turn a list of API results into an id -> object map, or to count word frequencies (as below).

In [3]:
doc_lengths = {f"doc_{i}": len(doc.split()) for i, doc in enumerate(DOCUMENTS)}
print(f"Doc-id -> word-count map: {doc_lengths}")

# Word frequency across ALL documents combined.
all_words = " ".join(DOCUMENTS).split()
word_freq: dict[str, int] = {}
for word in all_words:
    word_freq[word] = word_freq.get(word, 0) + 1
print(f"Word frequencies: {word_freq}")

top_5_words = dict(sorted(word_freq.items(), key=lambda kv: kv[1], reverse=True)[:5])
print(f"Top 5 most frequent words: {top_5_words}")

Doc-id -> word-count map: {'doc_0': 9, 'doc_1': 7, 'doc_2': 8, 'doc_3': 8}
Word frequencies: {'the': 4, 'quick': 1, 'brown': 1, 'fox': 2, 'jumps': 2, 'over': 1, 'lazy': 2, 'dog': 2, 'machine': 1, 'learning': 1, 'models': 2, 'learn': 1, 'patterns': 2, 'from': 2, 'data': 1, 'generative': 1, 'ai': 1, 'generate': 1, 'new': 1, 'content': 1, 'sleeps': 1, 'while': 1}
Top 5 most frequent words: {'the': 4, 'fox': 2, 'jumps': 2, 'lazy': 2, 'dog': 2}


## `*args` / `**kwargs` and unpacking

You'll see this constantly in API wrapper functions that need to accept flexible keyword arguments and forward them.

In [4]:
def summarize_stats(*numbers: float, **labels: str) -> dict:
    """Accepts any number of positional numbers and any number of
    keyword labels, and returns basic stats. Mirrors the shape of a
    real "call an API with flexible options" function."""
    return {
        "count": len(numbers),
        "min": min(numbers),
        "max": max(numbers),
        "avg": sum(numbers) / len(numbers),
        "labels": labels,
    }


lengths = [len(doc.split()) for doc in DOCUMENTS]
stats = summarize_stats(*lengths, dataset="demo_docs", unit="words")
print(f"Stats over word counts: {stats}")

first, *rest = DOCUMENTS  # unpacking: first element vs. the rest
print(f"\nFirst document: {first!r}")
print(f"Remaining {len(rest)} documents (truncated): {rest[0][:30]!r}...")

Stats over word counts: {'count': 4, 'min': 7, 'max': 9, 'avg': 8.0, 'labels': {'dataset': 'demo_docs', 'unit': 'words'}}

First document: 'the quick brown fox jumps over the lazy dog'
Remaining 3 documents (truncated): 'machine learning models learn '...


## A simple class

The reuse boundary you'll use for anything stateful, like a chat session that needs to remember conversation history.

In [5]:
class DocumentStore:
    """A minimal in-memory store, mirroring the shape of a real
    (much fancier) vector database client you'll meet in Track 3."""

    def __init__(self) -> None:
        self._docs: list[str] = []

    def add(self, text: str) -> None:
        self._docs.append(text)

    def search(self, keyword: str) -> list[str]:
        """Naive keyword search -- Track 3 replaces this with real
        semantic search over vector embeddings."""
        return [doc for doc in self._docs if keyword.lower() in doc.lower()]

    def __len__(self) -> int:
        return len(self._docs)


store = DocumentStore()
for doc in DOCUMENTS:
    store.add(doc)

print(f"Store now holds {len(store)} documents.")
results = store.search("fox")
print(f"Search results for 'fox': {results}")

Store now holds 4 documents.
Search results for 'fox': ['the quick brown fox jumps over the lazy dog', 'the lazy dog sleeps while the fox jumps']


### Try it yourself
Write a new function `top_n_words(text, n)` that returns the `n` most frequent words in a string, using a dict comprehension plus `sorted()`.